# 06 — Feature-engineered surface baseline

Measures how much of the task is solvable from basic surface statistics: 31 engineered
features, TF-IDF, and their concatenation, each tuned on the development split only.

Part-of-speech features are not included. No Kazakh POS tagger with characterised accuracy
on this text mixture was available, and an uncharacterised tagger would add an unmeasured
error source to a baseline whose purpose is to bound surface-level signal. Section 3 exposes
a hook for a tagger.

- Input: `data/cleaned/*.json`
- Output: `results_r2/table_A5_surface_baseline.csv`, `table_A5b_surface_feature_importance.csv`
- Runtime: ~2 min, CPU
- Reported in: paper Section VI-E

## 1. Config

In [ ]:
# CONFIG
REPO_DIR    = '.'
RESULTS_DIR = './results_r2'
SEED        = 42
BIN_WIDTH   = 5          # must match notebook 04 for the matched-subset column
ADD_POS_FEATURES = False # set True only if you supply a tagger in the hook cell

IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    REPO_DIR    = '/content/drive/MyDrive/kazakh-ai-text-detection'
    RESULTS_DIR = '/content/drive/MyDrive/kazakh-ai-text-detection/results_r2'

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'pandas', 'scipy', 'matplotlib'], check=False)

# Auto-load the shared paths written by notebook 00_setup_and_check.
# Run notebook 00 once and you never edit a path in any notebook again.
# If r2_config.json is absent, the values set above are used unchanged.
try:
    import json as _json
    from pathlib import Path as _Path
    _cfg_path = _Path('/content/drive/MyDrive/r2_config.json')
    if _cfg_path.exists():
        _cfg = _json.load(open(_cfg_path))
        REPO_DIR = _cfg['REPO_DIR']
        RESULTS_DIR = _cfg.get('RESULTS_DIR', RESULTS_DIR)
        print('Paths loaded from r2_config.json')
        print('  REPO_DIR   =', REPO_DIR)
        print('  RESULTS_DIR=', RESULTS_DIR)
    else:
        print('r2_config.json not found -- using the paths set above. '
              'Run 00_setup_and_check.ipynb to generate it.')
except Exception as _e:
    print('Could not load r2_config.json (%s: %s) -- using the paths set above.'
          % (type(_e).__name__, _e))


## 2. Load data

In [ ]:
import json, re
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

REPO = Path(REPO_DIR); OUT = Path(RESULTS_DIR); OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)
ID2LABEL = {0: 'Human', 1: 'AI-Generated', 2: 'AI-Obfuscated'}

def load_split(n):
    d = pd.DataFrame(json.load(open(REPO / f'data/cleaned/{n}_cleaned.json', encoding='utf-8')))
    d['text'] = d['text'].astype(str); d['label'] = d['label'].astype(int)
    d['n_words'] = d['text'].str.split().str.len()
    return d

train, dev, test = load_split('train'), load_split('dev'), load_split('test')
ytr, ydv, yte = (d['label'].to_numpy() for d in (train, dev, test))

# rebuild the length-matched index from notebook 04 (same rule, same seed)
t = test.copy(); t['bin'] = (t['n_words'] // BIN_WIDTH).astype(int)
keep = []
for b, g in t.groupby('bin'):
    vc = g['label'].value_counts()
    if len(vc) < 3:
        continue
    k = int(vc.min())
    for c in range(3):
        pool = g.index[g['label'] == c].to_numpy()
        keep.extend(rng.choice(pool, size=k, replace=False).tolist())
KEEP = np.array(sorted(keep))
print('Length-matched subset size:', len(KEEP))


## 3. Surface feature extractor (31 features; POS hook)

In [ ]:
PUNCT = list('.,!?;:-—()«»"\'…')

def surface_features(t):
    words = t.split(); nw = len(words); nc = len(t)
    sents = [s for s in re.split(r'[.!?…]+', t) if s.strip()]
    wl = [len(w) for w in words] or [0]
    toks = re.findall(r'\w+', t.lower(), flags=re.UNICODE)
    cnt = Counter(toks)
    cyr = sum('Ѐ' <= ch <= 'ӿ' for ch in t)
    lat = sum('a' <= ch.lower() <= 'z' for ch in t)
    f = {
        'n_words': nw, 'n_chars': nc, 'log_n_words': np.log1p(nw),
        'avg_word_len': float(np.mean(wl)), 'std_word_len': float(np.std(wl)),
        'n_sents': len(sents),
        'avg_sent_len_words': nw / len(sents) if sents else 0.0,
        'ttr': len(set(toks)) / len(toks) if toks else 0.0,
        'hapax_ratio': sum(1 for _, c in cnt.items() if c == 1) / len(toks) if toks else 0.0,
        'cyr_ratio': cyr / nc if nc else 0.0,
        'lat_ratio': lat / nc if nc else 0.0,
        'digit_ratio': sum(ch.isdigit() for ch in t) / nc if nc else 0.0,
        'upper_ratio': sum(ch.isupper() for ch in t) / nc if nc else 0.0,
        'long_word_ratio': sum(1 for w in wl if w > 10) / nw if nw else 0.0,
        'n_newlines': t.count('\n'),
        'space_ratio': t.count(' ') / nc if nc else 0.0,
    }
    for p in PUNCT:
        f[f'punct_{ord(p)}'] = t.count(p) / nc if nc else 0.0
    return f

# ---- POS HOOK -------------------------------------------------------
# Only enable this if you have a Kazakh POS tagger whose accuracy on this text
# type you can cite. A weak tagger makes the baseline worse AND less defensible.
def pos_features(t):
    raise NotImplementedError(
        'Plug in your Kazakh POS tagger here and return a dict of tag-frequency features, '
        'e.g. {"pos_NOUN": 0.31, "pos_VERB": 0.12, ...}. '
        'Remember to cite the tagger and report its accuracy in the paper.')

def featurize(df):
    rows = []
    for t in df['text']:
        f = surface_features(t)
        if ADD_POS_FEATURES:
            f.update(pos_features(t))
        rows.append(f)
    return pd.DataFrame(rows).astype(float)

Xtr, Xdv, Xte = featurize(train), featurize(dev), featurize(test)
print(f'{Xtr.shape[1]} features:', list(Xtr.columns))


## 4. Fit and select C on the development split

In [ ]:
def fit_select(Xtr_, ytr_, Xdv_, ydv_, grid, sparse=False):
    # Tune C on the development partition by macro-F1 -- never on test.
    best = None
    for C in grid:
        mdl = (LogisticRegression(max_iter=4000, class_weight='balanced', C=C) if sparse
               else make_pipeline(StandardScaler(),
                                  LogisticRegression(max_iter=4000, class_weight='balanced', C=C)))
        mdl.fit(Xtr_, ytr_)
        s = f1_score(ydv_, mdl.predict(Xdv_), average='macro')
        if best is None or s > best[0]:
            best = (s, C, mdl)
    return best

def report(name, mdl, Xte_, dev_f1, C):
    yp = mdl.predict(Xte_)
    per = f1_score(yte, yp, average=None, labels=[0, 1, 2])
    try:
        auc = roc_auc_score(yte, mdl.predict_proba(Xte_), multi_class='ovr', average='macro')
    except Exception:
        auc = np.nan
    return {'system': name, 'dev_macro_f1': round(dev_f1, 4), 'C': C,
            'test_macro_f1': round(f1_score(yte, yp, average='macro'), 4),
            'test_accuracy': round(accuracy_score(yte, yp), 4),
            'test_weighted_f1': round(f1_score(yte, yp, average='weighted'), 4),
            'roc_auc_ovr': round(float(auc), 4) if auc == auc else np.nan,
            'f1_Human': round(per[0], 4), 'f1_AI-Generated': round(per[1], 4),
            'f1_AI-Obfuscated': round(per[2], 4),
            'macro_f1_lengthmatched': round(f1_score(yte[KEEP], yp[KEEP], average='macro'), 4)}

results = []

# 1. length only
s, C, m = fit_select(Xtr[['n_words']], ytr, Xdv[['n_words']], ydv, [0.05, 0.25, 1, 5, 25])
results.append(report('Word count only', m, Xte[['n_words']], s, C))

# 2. surface features
s, C, m_surf = fit_select(Xtr, ytr, Xdv, ydv, [0.05, 0.25, 1, 5, 25])
results.append(report(f'Surface ({Xtr.shape[1]} features)', m_surf, Xte, s, C))

print(pd.DataFrame(results).to_string(index=False))


## 5. TF-IDF and TF-IDF + surface

In [ ]:
# 3. TF-IDF — reproduces the published baseline (sanity check vs notebook 03)
tf_w = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=10000)
tf_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3,
                       sublinear_tf=True, max_features=20000)
A = tf_w.fit_transform(train['text']); B = tf_c.fit_transform(train['text'])
Ttr = hstack([A, B]).tocsr()
Tdv = hstack([tf_w.transform(dev['text']),  tf_c.transform(dev['text'])]).tocsr()
Tte = hstack([tf_w.transform(test['text']), tf_c.transform(test['text'])]).tocsr()

s, C, m = fit_select(Ttr, ytr, Tdv, ydv, [0.25, 0.5, 1, 2, 5], sparse=True)
results.append(report('TF-IDF (word 1-2 + char 3-5)', m, Tte, s, C))

# 4. TF-IDF + surface
sc = StandardScaler().fit(Xtr)
Ctr = hstack([Ttr, csr_matrix(sc.transform(Xtr))]).tocsr()
Cdv = hstack([Tdv, csr_matrix(sc.transform(Xdv))]).tocsr()
Cte = hstack([Tte, csr_matrix(sc.transform(Xte))]).tocsr()
s, C, m = fit_select(Ctr, ytr, Cdv, ydv, [0.25, 0.5, 1, 2, 5], sparse=True)
results.append(report('TF-IDF + surface', m, Cte, s, C))

tbl = pd.DataFrame(results)
tbl.to_csv(OUT / 'table_A5_surface_baseline.csv', index=False)
print(tbl.to_string(index=False))

print('\nSanity check: the TF-IDF row should land near the published 0.776 macro-F1.')
print('A large discrepancy means the feature pipeline here differs from notebook 03 — '
      'reconcile before reporting.')


## 6. Feature importance

In [ ]:
# Which surface features carry the signal? (report the top few in the paper)
imp = pd.DataFrame({'feature': Xtr.columns,
                    'abs_coef_sum': np.abs(m_surf[-1].coef_).sum(0)}
                   ).sort_values('abs_coef_sum', ascending=False)
imp.to_csv(OUT / 'table_A5b_surface_feature_importance.csv', index=False)
print(imp.head(12).to_string(index=False))

print('\nIf length features (n_chars, n_words, n_sents) dominate, say so explicitly: it means the')
print('surface baseline is largely a length model, which is exactly why its macro-F1 collapses on')
print('the length-matched subset while the transformers barely move.')
